# Insurance Fraud Detection — Exploratory Data Analysis
**Dataset:** `insurance_fraud.xlsx` · 1000 rows × 39 columns  
**Target:** `fraud_reported` (Y = Fraud, N = Legitimate)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
PALETTE = {'Y': '#e74c3c', 'N': '#2ecc71'}

df = pd.read_excel('insurance_fraud.xlsx')
df['fraud_reported'] = df['fraud_reported'].map({'Y': 1, 'N': 0})

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

print(f'Shape: {df.shape}')
df.head(3)

## 1. Dataset Overview

In [ ]:
# Data types summary
type_summary = df.dtypes.value_counts().reset_index()
type_summary.columns = ['dtype', 'count']
print('Data Types:')
print(type_summary.to_string(index=False))

print('\nMissing Values:')
missing = df.isnull().sum()
missing = missing[missing > 0]
for col, cnt in missing.items():
    print(f'  {col}: {cnt} ({cnt/len(df)*100:.1f}%)')

print(f'\nDuplicate rows: {df.duplicated().sum()}')

## 2. Target Variable — Fraud Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['fraud_reported'].value_counts()
labels = ['Legitimate (N)', 'Fraud (Y)']
colors = ['#2ecc71', '#e74c3c']

# Bar chart
axes[0].bar(labels, [counts[0], counts[1]], color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Claim Count by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(
    [counts[0], counts[1]],
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: fraud_reported', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Fraud rate: {df["fraud_reported"].mean()*100:.1f}% | Class imbalance ratio: {counts[0]/counts[1]:.1f}:1')

## 3. Numerical Features — Distribution & Fraud Rate

In [ ]:
num_cols = [
    'months_as_customer', 'age', 'policy_annual_premium', 'policy_deductable',
    'capital-gains', 'capital-loss', 'total_claim_amount',
    'injury_claim', 'property_claim', 'vehicle_claim',
    'incident_hour_of_the_day', 'number_of_vehicles_involved',
    'bodily_injuries', 'witnesses', 'auto_year'
]

fig, axes = plt.subplots(5, 3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    fraud = df[df['fraud_reported'] == 1][col].dropna()
    legit = df[df['fraud_reported'] == 0][col].dropna()
    ax.hist(legit, bins=25, alpha=0.6, color='#2ecc71', label='Legitimate', density=True)
    ax.hist(fraud, bins=25, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions by Fraud Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for claim amounts
claim_cols = ['total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim']
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(claim_cols):
    data = [df[df['fraud_reported']==0][col].dropna(), df[df['fraud_reported']==1][col].dropna()]
    bp = axes[i].boxplot(data, patch_artist=True, labels=['Legitimate', 'Fraud'])
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    for patch in bp['boxes']:
        patch.set_alpha(0.7)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_ylabel('Amount ($)')

plt.suptitle('Claim Amounts by Fraud Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Categorical Features — Fraud Rate by Category

In [ ]:
cat_cols = [
    'incident_type', 'incident_severity', 'collision_type',
    'authorities_contacted', 'police_report_available', 'property_damage',
    'insured_sex', 'insured_education_level', 'insured_relationship'
]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    fraud_rate = df.groupby(col)['fraud_reported'].mean().sort_values(ascending=False)
    counts = df[col].value_counts()
    bars = axes[i].bar(range(len(fraud_rate)), fraud_rate.values, color='#3498db', alpha=0.75, edgecolor='white')
    axes[i].set_xticks(range(len(fraud_rate)))
    axes[i].set_xticklabels(fraud_rate.index, rotation=30, ha='right', fontsize=8)
    axes[i].set_ylabel('Fraud Rate')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=10)
    axes[i].axhline(df['fraud_reported'].mean(), color='red', linestyle='--', linewidth=1, alpha=0.7, label='Overall avg')
    axes[i].legend(fontsize=8)
    # Add count labels
    for j, (cat, rate) in enumerate(fraud_rate.items()):
        axes[i].text(j, rate + 0.005, f'n={counts.get(cat, 0)}', ha='center', fontsize=7)

plt.suptitle('Fraud Rate by Categorical Feature', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
corr_cols = num_cols + ['fraud_reported']
corr_matrix = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Full correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, ax=axes[0], mask=mask,
    annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    annot_kws={'size': 7},
    linewidths=0.5
)
axes[0].set_title('Correlation Matrix (Numerical Features)', fontweight='bold', fontsize=12)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', labelsize=8)

# Correlation with target
target_corr = corr_matrix['fraud_reported'].drop('fraud_reported').sort_values()
colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in target_corr]
axes[1].barh(target_corr.index, target_corr.values, color=colors, alpha=0.75, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Correlation with fraud_reported')
axes[1].set_title('Feature Correlation with Target', fontweight='bold', fontsize=12)
for i, v in enumerate(target_corr.values):
    axes[1].text(v + (0.003 if v >= 0 else -0.003), i, f'{v:.3f}',
                 va='center', ha='left' if v >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Claim Amount Deep Dive

In [ ]:
df['claim_to_premium_ratio'] = df['total_claim_amount'] / (df['policy_annual_premium'] + 1)
df['component_sum'] = df['injury_claim'] + df['property_claim'] + df['vehicle_claim']
df['claim_discrepancy'] = abs(df['total_claim_amount'] - df['component_sum'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Claim to premium ratio
for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
    vals = df[df['fraud_reported']==label]['claim_to_premium_ratio'].dropna()
    axes[0].hist(vals, bins=30, alpha=0.6, color=color,
                 label='Legitimate' if label==0 else 'Fraud', density=True)
axes[0].set_title('Claim-to-Premium Ratio', fontweight='bold')
axes[0].set_xlabel('Ratio')
axes[0].legend()

# Total claim distribution
bins = [0, 20000, 40000, 60000, 80000, 120000]
df['claim_bucket'] = pd.cut(df['total_claim_amount'], bins=bins,
                             labels=['<20k', '20-40k', '40-60k', '60-80k', '>80k'])
fraud_by_bucket = df.groupby('claim_bucket')['fraud_reported'].mean()
fraud_by_bucket.plot(kind='bar', ax=axes[1], color='#e67e22', alpha=0.8, rot=30)
axes[1].axhline(df['fraud_reported'].mean(), color='red', linestyle='--', label='Overall avg')
axes[1].set_title('Fraud Rate by Claim Amount Bucket', fontweight='bold')
axes[1].set_ylabel('Fraud Rate')
axes[1].legend()

# Component sum vs total
axes[2].scatter(
    df[df['fraud_reported']==0]['component_sum'],
    df[df['fraud_reported']==0]['total_claim_amount'],
    alpha=0.3, c='#2ecc71', s=10, label='Legitimate'
)
axes[2].scatter(
    df[df['fraud_reported']==1]['component_sum'],
    df[df['fraud_reported']==1]['total_claim_amount'],
    alpha=0.3, c='#e74c3c', s=10, label='Fraud'
)
lims = [0, 130000]
axes[2].plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='Equal line')
axes[2].set_xlabel('Sum of Components')
axes[2].set_ylabel('Total Claim Amount')
axes[2].set_title('Total vs Component Sum', fontweight='bold')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Incident Time & Severity Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fraud rate by hour of day
hourly_fraud = df.groupby('incident_hour_of_the_day')['fraud_reported'].mean()
axes[0].plot(hourly_fraud.index, hourly_fraud.values, marker='o', color='#e74c3c', linewidth=2)
axes[0].fill_between(hourly_fraud.index, hourly_fraud.values, alpha=0.2, color='#e74c3c')
axes[0].axhline(df['fraud_reported'].mean(), color='gray', linestyle='--', label='Overall avg')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Rate')
axes[0].set_title('Fraud Rate by Hour of Day', fontweight='bold')
axes[0].legend()
axes[0].set_xticks(range(0, 24, 2))

# Incident type × severity heatmap
pivot = df.pivot_table(
    values='fraud_reported',
    index='incident_type',
    columns='incident_severity',
    aggfunc='mean'
)
sns.heatmap(pivot, ax=axes[1], annot=True, fmt='.2f',
            cmap='Reds', linewidths=0.5, vmin=0, vmax=0.6)
axes[1].set_title('Fraud Rate: Type × Severity', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30, labelsize=8)
axes[1].tick_params(axis='y', rotation=0, labelsize=8)

# Number of vehicles vs fraud
vehicle_fraud = df.groupby('number_of_vehicles_involved')['fraud_reported'].agg(['mean', 'count'])
axes[2].bar(vehicle_fraud.index, vehicle_fraud['mean'], color='#9b59b6', alpha=0.75)
for x, (mean, count) in vehicle_fraud.iterrows():
    axes[2].text(x, mean + 0.005, f'n={int(count)}', ha='center', fontsize=9)
axes[2].axhline(df['fraud_reported'].mean(), color='red', linestyle='--', label='Overall avg')
axes[2].set_xlabel('Number of Vehicles Involved')
axes[2].set_ylabel('Fraud Rate')
axes[2].set_title('Fraud Rate by Vehicles Involved', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Customer Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
for label, color, name in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    vals = df[df['fraud_reported']==label]['age'].dropna()
    axes[0].hist(vals, bins=20, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('Age Distribution by Fraud Status', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].legend()

# Months as customer
df['tenure_bucket'] = pd.cut(df['months_as_customer'], bins=[0, 60, 120, 240, 360, 500],
                              labels=['<5yr', '5-10yr', '10-20yr', '20-30yr', '>30yr'])
tenure_fraud = df.groupby('tenure_bucket')['fraud_reported'].mean()
tenure_fraud.plot(kind='bar', ax=axes[1], color='#3498db', alpha=0.75, rot=30)
axes[1].axhline(df['fraud_reported'].mean(), color='red', linestyle='--', label='Overall avg')
axes[1].set_title('Fraud Rate by Customer Tenure', fontweight='bold')
axes[1].set_ylabel('Fraud Rate')
axes[1].legend()

# Hobbies fraud rate
hobby_fraud = df.groupby('insured_hobbies')['fraud_reported'].mean().sort_values(ascending=False).head(10)
hobby_fraud.plot(kind='barh', ax=axes[2], color='#e67e22', alpha=0.75)
axes[2].axvline(df['fraud_reported'].mean(), color='red', linestyle='--', label='Overall avg')
axes[2].set_title('Top 10 Hobbies by Fraud Rate', fontweight='bold')
axes[2].set_xlabel('Fraud Rate')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 9. Key Insights Summary

In [ ]:
print('=' * 65)
print('KEY FINDINGS')
print('=' * 65)

print(f'\n1. CLASS IMBALANCE')
print(f'   Fraud: {df["fraud_reported"].mean()*100:.1f}% | Legitimate: {(1-df["fraud_reported"].mean())*100:.1f}%')
print(f'   → Model needs class_weight=balanced or SMOTE')

print(f'\n2. CLAIM AMOUNTS')
fraud_total = df[df['fraud_reported']==1]['total_claim_amount'].mean()
legit_total = df[df['fraud_reported']==0]['total_claim_amount'].mean()
print(f'   Avg fraud claim: ${fraud_total:,.0f}')
print(f'   Avg legit claim: ${legit_total:,.0f}')
print(f'   → Fraud claims are {fraud_total/legit_total:.1f}x higher on average')

print(f'\n3. STRONGEST FRAUD PREDICTORS (correlation)')
target_corr = df[num_cols + ['fraud_reported']].corr()['fraud_reported'].drop('fraud_reported')
top3 = target_corr.abs().sort_values(ascending=False).head(3)
for feat, corr in top3.items():
    print(f'   {feat}: r={target_corr[feat]:.3f}')

print(f'\n4. INCIDENT SEVERITY')
sev_fraud = df.groupby('incident_severity')['fraud_reported'].mean().sort_values(ascending=False)
for sev, rate in sev_fraud.items():
    print(f'   {sev}: {rate*100:.1f}%')

print(f'\n5. HOBBIES FLAG')
high_risk_hobbies = df.groupby('insured_hobbies')['fraud_reported'].mean().sort_values(ascending=False).head(3)
for hobby, rate in high_risk_hobbies.items():
    print(f'   {hobby}: {rate*100:.1f}%')
print(f'   → Base jumping and skydiving are high-risk hobbies')

print(f'\n6. MISSING DATA')
print(f'   authorities_contacted: 91 nulls (9.1%)')
print(f'   collision_type / property_damage / police_report_available: use ? as placeholder')
print(f'   → Replace ? with NaN before modelling')

print('=' * 65)

## 10. Drift Test Data Generation
Generate 200 rows with **shifted numerical distributions** to test the drift detection pipeline.  
Categorical columns remain unchanged so only numerical PSI scores should breach the threshold.

In [ ]:
np.random.seed(42)
n = 200

# Use actual data for categorical columns (sample with replacement)
cat_cols_all = [
    'policy_state', 'policy_csl', 'insured_sex', 'insured_education_level',
    'insured_occupation', 'insured_hobbies', 'insured_relationship',
    'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted',
    'incident_state', 'incident_city', 'incident_location', 'property_damage',
    'police_report_available', 'auto_make', 'auto_model', 'fraud_reported'
]

drift_df = df[cat_cols_all].sample(n=n, replace=True).reset_index(drop=True)

# Add fake policy numbers and dates (non-model features)
import random, string
from datetime import datetime, timedelta

drift_df['policy_number'] = np.random.randint(100000, 999999, n)
drift_df['insured_zip']   = np.random.randint(420000, 499999, n)
drift_df['policy_bind_date'] = pd.to_datetime('2015-01-01') + pd.to_timedelta(np.random.randint(0, 1000, n), unit='D')
drift_df['incident_date']    = pd.to_datetime('2015-01-01') + pd.to_timedelta(np.random.randint(0, 1000, n), unit='D')

# ── SHIFTED numerical columns (PSI will be high) ──────────────────────────────

# total_claim_amount: mean shifts from ~52k to ~85k (high claims)
drift_df['total_claim_amount'] = np.random.normal(85000, 15000, n).clip(30000, 114920).astype(int)

# vehicle_claim: mean shifts from ~38k to ~65k
drift_df['vehicle_claim'] = np.random.normal(65000, 10000, n).clip(5000, 79560).astype(int)

# injury_claim: mean shifts from ~7.4k to ~14k
drift_df['injury_claim'] = np.random.normal(14000, 4000, n).clip(0, 21450).astype(int)

# property_claim: mean shifts from ~7.4k to ~13k
drift_df['property_claim'] = np.random.normal(13000, 4000, n).clip(0, 23670).astype(int)

# policy_annual_premium: mean shifts from ~1256 to ~1800
drift_df['policy_annual_premium'] = np.random.normal(1800, 150, n).clip(433, 2047).round(2)

# months_as_customer: mean shifts from ~204 to ~350 (older customers)
drift_df['months_as_customer'] = np.random.normal(350, 80, n).clip(0, 479).astype(int)

# age: mean shifts from ~39 to ~50
drift_df['age'] = np.random.normal(50, 8, n).clip(19, 64).astype(int)

# ── Stable numerical columns (PSI will be low) ───────────────────────────────
drift_df['policy_deductable']          = np.random.choice([500, 1000, 2000], n, p=[0.33, 0.34, 0.33])
drift_df['umbrella_limit']             = np.random.choice([-1000000, 0, 1000000, 2000000], n, p=[0.1, 0.5, 0.2, 0.2])
drift_df['capital-gains']              = np.random.choice([0, 25000, 50000, 75000, 100500], n, p=[0.4, 0.2, 0.2, 0.1, 0.1])
drift_df['capital-loss']               = np.random.choice([0, -25000, -50000, -75000, -111100], n, p=[0.4, 0.2, 0.2, 0.1, 0.1])
drift_df['incident_hour_of_the_day']   = np.random.randint(0, 24, n)
drift_df['number_of_vehicles_involved']= np.random.choice([1, 2, 3, 4], n, p=[0.4, 0.35, 0.15, 0.1])
drift_df['bodily_injuries']            = np.random.choice([0, 1, 2], n, p=[0.33, 0.34, 0.33])
drift_df['witnesses']                  = np.random.choice([0, 1, 2, 3], n, p=[0.25, 0.35, 0.25, 0.15])
drift_df['auto_year']                  = np.random.randint(1995, 2016, n)

# Save
drift_df.to_excel('drift_test_data.xlsx', index=False)
print(f'Drift test data saved: {drift_df.shape}')
print(f'\nColumns: {drift_df.columns.tolist()}')
drift_df.head(3)

In [ ]:
# Visualise the shift in key numerical columns
shifted_cols = ['total_claim_amount', 'vehicle_claim', 'injury_claim', 'policy_annual_premium', 'months_as_customer', 'age']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(shifted_cols):
    axes[i].hist(df[col].dropna(), bins=25, alpha=0.6, color='#3498db', density=True, label='Training data')
    axes[i].hist(drift_df[col].dropna(), bins=25, alpha=0.6, color='#e74c3c', density=True, label='Drift data')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Training Data vs Drift Test Data — Shifted Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('These 6 columns will produce high PSI scores when fed to the drift detector.')
print('Categorical columns are sampled from the original data → PSI will be low.')